<a href="https://colab.research.google.com/github/asema405/Maachine_Learning/blob/main/ML_PR3_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

print("Все библиотеки успешно импортированы!")

Все библиотеки успешно импортированы!


In [2]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"

df = pd.read_csv(url)

print("Размер исходного датасета:", df.shape)
display(df.head())

Размер исходного датасета: (244, 7)


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [3]:
df_raw = df.copy()

# Создаём искусственные пропуски для демонстрации предобработки
df_raw.loc[[2, 10, 25], "total_bill"] = np.nan
df_raw.loc[[5, 15, 30], "tip"] = np.nan
df_raw.loc[[8, 18], "sex"] = np.nan
df_raw.loc[[12, 22], "day"] = np.nan

print("Размер датасета:", df_raw.shape)
print("\nКоличество пропусков по столбцам:")
display(df_raw.isnull().sum())

Размер датасета: (244, 7)

Количество пропусков по столбцам:


,0
total_bill,3
tip,3
sex,2
smoker,0
day,2
time,0
size,0


In [4]:
numeric_features = df_raw.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = df_raw.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Числовые признаки:", numeric_features)
print("Категориальные признаки:", categorical_features)

Числовые признаки: ['total_bill', 'tip', 'size']
Категориальные признаки: ['sex', 'smoker', 'day', 'time']


In [5]:
X_train, X_test = train_test_split(
    df_raw,
    test_size=0.2,
    random_state=42
)

print("Размер X_train:", X_train.shape)
print("Размер X_test:", X_test.shape)

Размер X_train: (195, 7)
Размер X_test: (49, 7)


In [6]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

print("Числовой pipeline создан.")

Числовой pipeline создан.


In [7]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="неизвестно"
            )
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

print("Категориальный pipeline создан.")

Категориальный pipeline создан.


In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("ColumnTransformer успешно создан.")

ColumnTransformer успешно создан.


In [9]:
preprocessor.fit(X_train)

print("Пайплайн обучен только на тренировочных данных.")

Пайплайн обучен только на тренировочных данных.


In [10]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Размер обработанного train:", X_train_processed.shape)
print("Размер обработанного test:", X_test_processed.shape)

Размер обработанного train: (195, 15)
Размер обработанного test: (49, 15)


In [11]:
print("Количество признаков train:", X_train_processed.shape[1])
print("Количество признаков test:", X_test_processed.shape[1])

assert X_train_processed.shape[1] == X_test_processed.shape[1]

print("Проверка пройдена: размерности train и test совпадают.")

Количество признаков train: 15
Количество признаков test: 15
Проверка пройдена: размерности train и test совпадают.


In [12]:
print("Форма train:", X_train_processed.shape)
print("Форма test:", X_test_processed.shape)

if hasattr(X_train_processed, "toarray"):
    train_array = X_train_processed.toarray()
    test_array = X_test_processed.toarray()
else:
    train_array = np.asarray(X_train_processed)
    test_array = np.asarray(X_test_processed)

print("Есть ли NaN в train:", np.isnan(train_array).any())
print("Есть ли NaN в test:", np.isnan(test_array).any())

Форма train: (195, 15)
Форма test: (49, 15)
Есть ли NaN в train: False
Есть ли NaN в test: False


## Обоснование выбора стратегий предобработки

1. **Числовые признаки (`total_bill`, `tip`, `size`).** Для заполнения пропусков выбрана медиана (`SimpleImputer(strategy="median")`). Медиана менее чувствительна к выбросам, чем среднее арифметическое.

2. **Масштабирование числовых признаков.** Использован `StandardScaler`, который приводит числовые признаки к сопоставимому масштабу. Параметры масштабирования вычисляются только по тренировочной выборке.

3. **Категориальные признаки (`sex`, `smoker`, `day`, `time`).** Пропуски заменяются отдельной категорией «неизвестно». Это позволяет не удалять строки и явно обозначать отсутствие информации.

4. **Кодирование категорий.** Применён `OneHotEncoder(handle_unknown="ignore")`. Он преобразует категории в числовые столбцы и позволяет обрабатывать новые категории в тестовой выборке.

5. **Объединение преобразований.** Использован `ColumnTransformer`, который применяет разные операции к числовым и категориальным признакам. Все шаги объединены в `Pipeline`.

6. **Предотвращение утечки данных.** Сначала данные разделены на тренировочную и тестовую выборки. Метод `fit()` вызван только для `X_train`, а затем `transform()` применён к обеим выборкам.

7. **Проверка результата.** После предобработки количество признаков в тренировочной и тестовой матрицах совпадает. Дополнительно проверено отсутствие пропущенных числовых значений в обработанных матрицах.

### Вывод

В ходе практической работы создан пайплайн предобработки данных с использованием `Pipeline`, `ColumnTransformer`, `SimpleImputer`, `StandardScaler` и `OneHotEncoder`. Пайплайн обучен только на тренировочной выборке и применён к тестовой без повторного обучения, что помогает предотвратить утечку данных.

In [13]:
from pathlib import Path

Path("data").mkdir(exist_ok=True)
Path("notebooks").mkdir(exist_ok=True)

df_raw.to_csv("data/tips_raw.csv", index=False)

readme = """# Практическая работа № 3. Сборка пайплайна предобработки

ФИО: Ашимова Асема Дамировна

Вариант: B

Датасет: Tips — данные о счетах и чаевых в ресторане.

Среда выполнения: Google Colab.

Выполненные задания:
- Классификация числовых и категориальных признаков.
- Разделение данных на train и test.
- Заполнение числовых пропусков медианой.
- Заполнение категориальных пропусков отдельной категорией.
- Масштабирование числовых признаков.
- One-hot кодирование категориальных признаков.
- Объединение преобразований через ColumnTransformer и Pipeline.
- Проверка размерностей и отсутствия утечки данных.

Примечание: пропущенные значения были искусственно добавлены
для демонстрации работы пайплайна.
"""

Path("README.md").write_text(readme, encoding="utf-8")

print("Файлы README.md и data/tips_raw.csv сохранены.")

Файлы README.md и data/tips_raw.csv сохранены.
